# Stochastic noise lab

The most varied synthetic family in the project. Every clip is a realization of
a random spectrum: a smooth colored floor, plus one Lorentzian line per harmonic
of every rotor, with **every amplitude drifting slowly in time as a Gaussian
process**.

The model is `data_processing.stochastic_rotor_noise`; this notebook only drives
it. It is the generative direction of the v4 analysis model
(`tracking.joint_decompose`), which reads a real recording as a smooth floor
with sparse Lorentzian lines on it.

What each control does:

* **harmonic level** and **broadband level** — the two amplitude means, in dB.
  Their difference is how far the comb stands above the floor.
* **harmonic wander** and **broadband wander** — the standard deviation of the
  Gaussian process on each amplitude, in dB. Zero freezes the amplitudes.
* **wander time** — the correlation time of the same process, in seconds. Short
  makes the lines flicker; long makes the clip drift as a whole.
* **harmonic coherence** — how much of a rotor's wander is shared by all its
  harmonics. At 0 every line breathes alone; at 1 the whole comb breathes
  together.
* **floor color wander** — how much the floor's tilt moves, in dB per octave.

Rotor speeds come from the OU model, the intermittent (pilot and airframe)
model, a whole flight from the ground and back, or the telemetry of a real
recording.

**New random parameters** draws a new drone: a new timbre, a new floor color, a
new set of linewidths. **Regenerate** keeps the drone and re-renders with the
current slider values and a new trajectory.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
for p in (ROOT, ROOT / "src", ROOT / "notebooks"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import warnings

import matplotlib.pyplot as plt
import numpy as np

from stochastic_noise_lab import Lab

warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams.update({"figure.dpi": 110})

## 1 · The panel

Sample, listen, look, move a slider, regenerate.

In [2]:
lab = Lab(seed=0)
lab.panel()

## 2 · What the current clip is made of

The scalar parameters of whatever the panel rendered last. The static random
parts — the per-rotor harmonic profile, the floor's shape curve, the per-rotor
linewidths — are arrays and are left out of this readout; plot them below.

In [ ]:
lab.summary()

In [ ]:
params = lab.last["params"]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
for r in range(params.n_rotors):
    axes[0].plot(np.arange(1, params.n_harmonics + 1), params.profile_db[r], lw=1.0,
                 label=f"rotor {r + 1}")
axes[0].set_xlabel("harmonic")
axes[0].set_ylabel("dB")
axes[0].set_title("harmonic amplitude profiles")
axes[0].legend(fontsize=7)
axes[0].grid(alpha=0.2)

from data_processing import stochastic_rotor_noise as srn

freqs = np.linspace(30.0, 8000.0, 2000)
axes[1].semilogx(freqs, srn.floor_shape_db(params, freqs), lw=1.4, color="#c0392b")
axes[1].set_xlabel("frequency (Hz)")
axes[1].set_ylabel("dB")
axes[1].set_title("broadband shape")
axes[1].grid(alpha=0.2)
fig.tight_layout()

## 3 · How much the amplitudes moved

The Gaussian-process draws of the last render. These are what the covariance
sliders control, and they are drawn independently of the rotor speeds — which
is why a predictor cannot read speed off any amplitude in this family.

In [ ]:
diag = lab.last["diag"]
t = diag["frame_times"]
fig, axes = plt.subplots(2, 1, figsize=(11, 4.2), sharex=True)
for k in range(0, min(diag["harm_gp"].shape[1], 24), 3):
    axes[0].plot(t, diag["harm_gp"][0, k], lw=0.9, label=f"k={k + 1}")
axes[0].set_ylabel("dB")
axes[0].set_title("rotor 1 harmonic amplitude processes")
axes[0].legend(fontsize=6, ncol=8)
axes[0].grid(alpha=0.2)
axes[1].plot(t, diag["floor_gp"], lw=1.2, color="#c0392b", label="floor level (dB)")
axes[1].plot(t, diag["floor_tilt_gp"], lw=1.2, color="#1f5fa9", label="floor tilt (dB/oct)")
axes[1].set_xlabel("time (s)")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.2)
fig.tight_layout()

## 4 · The family as a training stream

`StochasticNoisePool` is the online-mix source (`kind: stochastic`): every
window draws a fresh parameter set, so no two windows share a timbre. This is
the same interface the static-comb and generated pools use, so it drops into a
policy YAML next to them.

In [ ]:
from data_processing.stochastic_rotor_noise import StochasticNoisePool

pool = StochasticNoisePool(sample_rate=16000, duration_s=4.0, n_mics=8,
                           rps_kind="full_flight")
rng = np.random.default_rng(0)

fig, axes = plt.subplots(1, 4, figsize=(15, 3.0))
for ax in axes:
    frame = pool.sample_timeframe(rng, 4.0)
    x = np.asarray(frame["audio"].data)[0].astype(np.float64)
    n_fft, hop = 1024, 256
    w = np.hanning(n_fft + 1)[:n_fft]
    fr = np.stack([x[i * hop:i * hop + n_fft] * w for i in range((x.size - n_fft) // hop)])
    db = 10 * np.log10(np.maximum(np.abs(np.fft.rfft(fr, axis=-1)) ** 2, 1e-20)).T
    ax.imshow(db, origin="lower", aspect="auto", cmap="magma",
              vmin=db.max() - 70, vmax=db.max(),
              extent=[0, 4, 0, 8000])
    ax.set_ylim(0, 6000)
    ax.set_xlabel("time (s)")
axes[0].set_ylabel("frequency (Hz)")
fig.suptitle("four windows of the stream — four different drones")
fig.tight_layout()